# Preprocess code 3 - C retailer
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text

Load data:

In [ ]:
data = pd.read_csv(wd_dp + "C_raw.csv")
len(data)

In [ ]:
data = data.rename(columns={"Descripcion": "descripcion", "Unidades": "unidades"})

Join columns with same meaning

In [ ]:
data["precio"] = data['precio'].fillna(data['Precio'])
data["precio_unidad"] = data['Precio_mililitro'].fillna(data['precio_und'])
data["fecha"] = data['fecha'].fillna(data['Fecha'])
data["link"] = data['link'].fillna(data['Link'])

Filter empty products

In [ ]:
data = data[~data['descripcion'].isna()]
data = data[~data['precio'].isna()]
len(data)

Fill searched word 

In [ ]:
data['palabra'] = data['palabra'].fillna(data['Link'].str.extract(r"https?://[^/]+/search\?name=(.*)").squeeze())

String homogenize

In [ ]:
data["palabra"] = data["palabra"].apply(homogenize_text)
data["descripcion"] = data["descripcion"].apply(homogenize_text)

Price homogenize

In [ ]:
data["precio"] = data["precio"].astype("str")
data["precio"] = data["precio"].str.replace("\.0$", "", regex=True)
data["precio"] = data["precio"].str.replace("[a-zA-Z]", "", regex=True)
data["precio"] = data["precio"].str.replace(".", "")
data["precio"] = data["precio"].str.replace(",", ".")
data["precio"] = data["precio"].str.replace("$", "")

Filter empty price products

In [ ]:
data = data[data["precio"].str.len()!=0]
data = data[~data['precio'].isna()]

Price to numeric

In [ ]:
data["precio"] = data["precio"].astype("float")

In [ ]:
data.columns

Select columns

In [ ]:
data = data[["fecha", "descripcion", "precio", "palabra"]]
data["tienda"] = "C"

Save preprocess data

Summarize duplicate prices

In [ ]:
data = data.groupby(['fecha', 'descripcion', 'tienda']).agg({'precio': 'min'}).reset_index()

Show data

In [ ]:
display(data)

In [ ]:
pd.DataFrame(data.to_csv(wd_db+"C_clean.csv"))